# B6 · Explicabilité, équité et éthique — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b6_explicabilite/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B6.Q1 à B6.Q12, B6.R1 à B6.R3, B6.1, B6.3 à B6.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B6.12 à B6.17 | les modèles tabulaires : explications globales sur California, Shapley exact avec ta librairie, shap sur un gradient boosting, le rôle du fond, Penguins (*prochaine session de génération*) | | | |
| B | B6.18 à B6.22 | les réseaux : gradients intégrés et Grad-CAM dans ta librairie, attributions sur un CNN CIFAR-10, Grad-CAM par hooks sur le ResNet du B1, un Grad-CAM à réparer, le test de santé (*prochaine session de génération*) | | | |
| C | B6.23 à B6.29 | l'équité : tes indicateurs par groupe, l'audit d'Adult avec intervalles, l'effet du retrait du sexe, l'atténuation et son coût, la fiche modèle et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main des valeurs de Shapley exactes, des gradients intégrés et une carte Grad-CAM, et les relier à ta librairie (`explain.py`) et à shap.
- Expliquer un modèle tabulaire globalement (permutation, dépendance partielle, ICE) et localement (Shapley, waterfall), en disant quel fond a servi.
- Produire des cartes d'attribution sur un CNN et les soumettre à un test de santé.
- Auditer un classifieur groupe par groupe, avec des intervalles, essayer des corrections, chiffrer leur coût et documenter le tout dans une fiche modèle.

**Rappel express.** Shapley : $v(S) = \frac{1}{B}\sum_k f(\mathbf{x}_S, \mathbf{b}^{(k)}_{\bar S})$, $\phi_i = \sum_{S \not\ni i} \frac{|S|!\,(M - |S| - 1)!}{M!}\,[v(S \cup \{i\}) - v(S)]$, $\sum_i \phi_i = f(\mathbf{x}) - v(\varnothing)$ ; modèle linéaire : $\phi_i = w_i(x_i - \bar b_i)$. Gradients intégrés : $\operatorname{IG}_i = (x_i - x'_i)\int_0^1 \partial_i f(\mathbf{x}' + \alpha(\mathbf{x} - \mathbf{x}'))\,d\alpha$, de somme $f(\mathbf{x}) - f(\mathbf{x}')$. Grad-CAM : $\alpha_c$ = moyenne spatiale de $\partial y^k / \partial A^c$, carte $\operatorname{ReLU}(\sum_c \alpha_c A^c)$. Équité : $\Delta_{\text{DP}}$ = écart des taux de sélection, $\Delta_{\text{EO}}$ = le plus grand des écarts de TPR et de FPR ; $\operatorname{FPR} = \frac{p}{1-p}\cdot\frac{1 - \operatorname{PPV}}{\operatorname{PPV}}\cdot(1 - \operatorname{FNR})$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B6.1, B6.3, B6.4 et B6.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un vecteur : une liste de nombres (`[0.5, -1.25]`). Un taux s'écrit comme une proportion (`0.25`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ B6.2 et B6.6, l'estimation 🧮 B6.7, la réflexion (🗣️ B6.8, ⚖️ B6.9 et B6.10, 📄 B6.11) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md). Features are numbered 1..M in
# the formulas, columns 0..M-1 in the code (feature 1 is column 0).
import itertools
import math

import numpy as np


def coalition_values(f, x, background):
    """v(S) for every mask S (a tuple of 0/1): the mean of f over the background rows where S takes x's values."""
    x, background = np.asarray(x, float), np.asarray(background, float)
    values = {}
    for mask in itertools.product((0, 1), repeat=len(x)):
        rows = background.copy()
        keep = np.array(mask, bool)
        rows[:, keep] = x[keep]
        values[mask] = float(np.mean(f(rows)))
    return values


def shapley_from_values(values, M, weight=None):
    """phi_i = sum over S without i of weight(|S|) (v(S + i) - v(S)); weight defaults to |S|! (M-|S|-1)! / M!."""
    weight = weight or (lambda k: math.factorial(k) * math.factorial(M - k - 1) / math.factorial(M))
    phi = []
    for i in range(M):
        total = 0.0
        for mask, value in values.items():
            if not mask[i]:
                with_i = tuple(1 if j == i else mask[j] for j in range(M))
                total += weight(sum(mask)) * (values[with_i] - value)
        phi.append(total)
    return phi


# ✏️ B6.1: f(x) = 2 x1 + x1 x3 + x2, x = (2, 1, 3), background (0, 2, 0) and (2, 2, 2)
F_B61 = lambda X: 2 * X[:, 0] + X[:, 0] * X[:, 2] + X[:, 1]
X_B61, BG_B61 = np.array([2.0, 1.0, 3.0]), np.array([[0.0, 2.0, 0.0], [2.0, 2.0, 2.0]])
V_B61 = coalition_values(F_B61, X_B61, BG_B61)
V0_B61, V1_B61, V13_B61 = V_B61[(0, 0, 0)], V_B61[(1, 0, 0)], V_B61[(1, 0, 1)]
W3_B61 = math.factorial(1) * math.factorial(1) / math.factorial(3)        # |S| = 1, M = 3
PHI_B61 = shapley_from_values(V_B61, 3)
FIRST_B61 = [V_B61[tuple(int(j == i) for j in range(3))] - V0_B61 for i in range(3)]   # contribution to the empty set
LAST_B61 = [V_B61[(1, 1, 1)] - V_B61[tuple(int(j != i) for j in range(3))] for i in range(3)]
EVALS_B61 = 2 ** 3 * len(BG_B61)
MEAN_ROW_B61 = BG_B61.mean(axis=0, keepdims=True)                          # (1, 2, 1)
PHI_MEAN_B61 = shapley_from_values(coalition_values(F_B61, X_B61, MEAN_ROW_B61), 3)
INTER_SHARE_B61 = [PHI_B61[0] - 2 * (X_B61[0] - BG_B61[:, 0].mean()), PHI_B61[2]]   # the share of x1 x3 in phi_1, phi_3

# ✏️ B6.3: f(x) = x1**2 + 3 x2 at x = (2, -1), zero baseline unless stated otherwise
X_B63 = np.array([2.0, -1.0])
F_B63 = lambda P: P[..., 0] ** 2 + 3 * P[..., 1]
GRAD_B63 = lambda P: np.stack([2 * P[..., 0], np.full(np.shape(P)[:-1], 3.0)], axis=-1)


def ig_rule_b63(n, method, x=X_B63, ref=(0.0, 0.0)):
    """Integrated gradients with n intervals (the conventions of the docstring of integrated_gradients)."""
    x, ref = np.asarray(x, float), np.asarray(ref, float)
    k = np.arange(n + 1) / n
    alphas = {"left": k[:-1], "right": k[1:], "midpoint": (np.arange(n) + 0.5) / n, "trapezoid": k}[method]
    weights = np.full(len(alphas), 1 / n)
    if method == "trapezoid":
        weights[[0, -1]] /= 2
    grads = GRAD_B63(ref + alphas[:, None] * (x - ref))
    return (x - ref) * (weights[:, None] * grads).sum(axis=0)


GRAD_AT_X_B63 = [float(g) for g in GRAD_B63(X_B63)]
GXI_B63 = [float(g) for g in GRAD_B63(X_B63) * X_B63]
GAP_B63 = float(F_B63(X_B63) - F_B63(np.zeros(2)))
IG_EXACT_B63 = [float(X_B63[0] ** 2), float(3 * X_B63[1])]                 # x1 * integral of 2 a x1 = x1**2 ; 3 x2
IG1_LEFT_B63, IG1_RIGHT_B63, IG1_TRAP_B63 = (float(ig_rule_b63(5, m)[0]) for m in ("left", "right", "trapezoid"))
N_LEFT_B63 = math.ceil(round(X_B63[0] ** 2 / 0.03, 9))                     # the left error of IG_1 is x1**2 / n
REF2_B63 = (1.0, 0.0)
IG1_REF2_B63 = float(X_B63[0] ** 2 - REF2_B63[0] ** 2)                     # (x1 - x1') (x1 + x1')
SUM_REF2_B63 = float(F_B63(X_B63) - F_B63(np.array(REF2_B63)))
CAPTUM_B63 = IG1_TRAP_B63 * 5 / 6                                          # 6 points, weights 1/6 halved at both ends

# ✏️ B6.4: two 3 x 3 channels (index 0 = channel 1)
A_B64 = np.array([[[1, 0, 0], [2, 1, 0], [3, 2, 1]], [[0, 1, 2], [0, 0, 1], [0, 0, 0]]], dtype=float)
G_B64 = np.array([[[0, 1, 2], [1, 1, 1], [2, 1, 0]], [[-1, -1, -1], [0, 0, 0], [-1, -0.5, 0]]], dtype=float)
ALPHA_B64 = G_B64.mean(axis=(1, 2))
RAW_B64 = np.tensordot(ALPHA_B64, A_B64, axes=1)                           # before the ReLU
CAM_B64 = np.maximum(RAW_B64, 0)
NORM_B64 = CAM_B64 / np.abs(CAM_B64).max()
TOPRIGHT_B64 = float(RAW_B64[0, 2])
ZEROS_B64 = int((CAM_B64 == 0).sum())
CENTER_B64 = float(NORM_B64[1, 1])
NORELU_TOPRIGHT_B64 = float(RAW_B64[0, 2] / np.abs(RAW_B64).max())
OTHER_B64 = np.tensordot(np.array([-0.5, 1.0]), A_B64, axes=1)             # class k': alpha' = (-0.5, 1)
CORNERS_B64 = {(0, 0): "A", (2, 0): "B", (0, 2): "C"}
MAX_B64 = CORNERS_B64[tuple(int(i) for i in np.unravel_index(np.argmax(CAM_B64), CAM_B64.shape))]
MAX_OTHER_B64 = CORNERS_B64[tuple(int(i) for i in np.unravel_index(np.argmax(OTHER_B64), OTHER_B64.shape))]
SHIFT_B64 = 72 / 3 / 2                                                     # half a layer pixel, in image pixels

# ✏️ B6.5: two groups (TP, FN, FP, TN)
COUNTS_B65 = {"F": (72, 48, 19, 361), "H": (256, 64, 40, 280)}


def rates_b65(tp, fn, fp, tn):
    n = tp + fn + fp + tn
    return {"base": (tp + fn) / n, "sel": (tp + fp) / n, "tpr": tp / (tp + fn), "fpr": fp / (fp + tn),
            "ppv": tp / (tp + fp), "acc": (tp + tn) / n}


R_B65 = {g: rates_b65(*c) for g, c in COUNTS_B65.items()}
DP_B65 = abs(R_B65["H"]["sel"] - R_B65["F"]["sel"])
TPR_GAP_B65 = abs(R_B65["H"]["tpr"] - R_B65["F"]["tpr"])
FPR_GAP_B65 = abs(R_B65["H"]["fpr"] - R_B65["F"]["fpr"])
EO_B65 = max(TPR_GAP_B65, FPR_GAP_B65)
RATIO_B65 = R_B65["F"]["sel"] / R_B65["H"]["sel"]
ACC_GAP_B65 = abs(R_B65["F"]["acc"] - R_B65["H"]["acc"])
POS_F_B65 = COUNTS_B65["F"][0] + COUNTS_B65["F"][1]
EXTRA_B65 = round(R_B65["H"]["tpr"] * POS_F_B65 - COUNTS_B65["F"][0])

# quizzes and recalls with a computation
Q1E = 2 ** 4
Q2E = 0.83 - float(np.mean([0.66, 0.62, 0.65, 0.63, 0.64]))
Q3C = 3 * 2 + float(np.mean(np.array([0, 1, 2]) ** 2))
Q4A = 0.72 - 0.40 - 0.15 - (-0.05)
Q4C = 0.3 + (-0.1)
Q4E = 2 ** (5 - 1)
Q5C, Q5D = 2 * (4 - 3), 2 * (4 - 5)
Q6A, Q6B = 2 * (1.5 - 0.5), 0 * (1.5 - 0.5)
Q7A = 0                                                     # min(x, 1) is flat beyond 1
Q7B = 3 * (1 / 3)                                           # (x - 0) * integral of f'(3 a): f' = 1 for a < 1/3
Q7C = 0                                                     # the path from 2 to 3 stays where f is flat
Q8A, Q8C = 160 // 32, 224 // 4
Q9E = 1 / 10
Q11A = 0.6 - 0.25                                           # a perfect classifier selects exactly the positives
Q11D = 0.5 / (1 - 0.5) * (1 - 0.8) / 0.8 * (1 - 0.2)        # Chouldechova
EPS_R1, X_R1, G_R1 = 0.03, np.array([0.40, 0.98, 0.10]), np.array([-0.7, 0.2, 0.0])
R1A = [float(v) for v in np.clip(X_R1 + EPS_R1 * np.sign(G_R1), 0, 1)]
R1D = 0.01 * 3072 * 0.01
R2E = round((0.88 - 0.86) * 400)
CM_R3 = np.array([[88, 12], [10, 40]])                      # rows: truth 0, 1 ; columns: prediction 0, 1
(TN_R3, FP_R3), (FN_R3, TP_R3) = CM_R3
R3A, R3B = TP_R3 / (TP_R3 + FN_R3), FP_R3 / (FP_R3 + TN_R3)
R3C, R3D = TP_R3 / (TP_R3 + FP_R3), (TP_R3 + FP_R3) / CM_R3.sum()

# The same answers from the libraries (no download): mylearn (reference), shap, scikit-learn and torchvision
E = mylearn.explain

# ✏️ B6.1: shapley_values, then shap's exact explainer with every background row
phi, base = E.shapley_values(F_B61, X_B61, BG_B61)
assert np.allclose(phi, PHI_B61) and math.isclose(base, V0_B61)
assert math.isclose(phi.sum() + base, float(F_B61(X_B61[None])[0]))       # efficiency
assert np.allclose(E.shapley_values(F_B61, X_B61, MEAN_ROW_B61)[0], PHI_MEAN_B61)
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm, imported by shap
try:
    import shap
except ImportError:
    print("ℹ️ shap is not installed: the comparison with shap.explainers.Exact is skipped")
else:
    exact = shap.explainers.Exact(F_B61, shap.maskers.Independent(BG_B61, max_samples=len(BG_B61)))(X_B61[None])
    assert np.allclose(exact.values[0], PHI_B61) and math.isclose(float(exact.base_values[0]), V0_B61)

# quiz Q4 to Q6: the linear models of the statements (phi_i = w_i (x_i - mean_i))
lin = lambda w: (lambda X: X @ np.asarray(w, float))
assert np.allclose(E.shapley_values(lin([2, -3]), [4.0, 1.0], [[3.0, 1.0]])[0], [Q5C, -3 * 0])
assert math.isclose(E.shapley_values(lin([2, -3]), [4.0, 1.0], [[4.0, 1.0], [6.0, 3.0]])[0][0], Q5D)
phi_a = E.shapley_values(lin([2, 2]), [1.5, 1.5], [[0.0, 0.0], [1.0, 1.0]])[0]
phi_b = E.shapley_values(lin([4, 0]), [1.5, 1.5], [[0.0, 0.0], [1.0, 1.0]])[0]
assert math.isclose(phi_a[0], Q6A) and math.isclose(phi_b[1], Q6B) and math.isclose(phi_a.sum(), phi_b.sum())
# ∂ B6.2, question 5 (an application, not checked by wb.check)
phi_d2, base_d2 = E.shapley_values(lambda X: 1 + X @ np.array([0.5, -2.0, 0.0]), [4.0, 1.0, 7.0],
                                   [[2.0, 0.0, 1.0], [4.0, 1.0, 5.0], [0.0, 2.0, 3.0]])
assert np.allclose(phi_d2, [1.0, 0.0, 0.0]) and math.isclose(base_d2, 0.0, abs_tol=1e-12)

# ✏️ B6.3: integrated_gradients with the rules of its docstring
ig = lambda **kw: E.integrated_gradients(GRAD_B63, X_B63, **kw)
for method, answer in (("left", IG1_LEFT_B63), ("right", IG1_RIGHT_B63), ("trapezoid", IG1_TRAP_B63)):
    assert math.isclose(ig(n_steps=5, method=method)[0], answer)
assert np.allclose(ig(n_steps=5, method="trapezoid"), IG_EXACT_B63)       # exact: the gradient is linear on the path
assert abs(GAP_B63 - ig(n_steps=N_LEFT_B63, method="left").sum()) <= 0.03 - 1e-4      # far from the edge (4/134)
assert abs(GAP_B63 - ig(n_steps=N_LEFT_B63 - 1, method="left").sum()) > 0.03 + 1e-5
assert np.allclose(ig(baseline=list(REF2_B63), n_steps=3), [IG1_REF2_B63, IG_EXACT_B63[1]])
assert math.isclose(ig(baseline=list(REF2_B63), n_steps=3).sum(), SUM_REF2_B63, abs_tol=1e-12)
alphas = np.linspace(0, 1, 6)                                             # captum's riemann_trapezoid, 6 points
weights = np.full(6, 1 / 6)
weights[[0, -1]] /= 2
assert math.isclose(X_B63[0] * float(weights @ GRAD_B63(alphas[:, None] * X_B63)[:, 0]), CAPTUM_B63)

# quiz Q7: min(x, 1) at x = 3 (its gradient: 1 below 1, 0 beyond)
grad_min = lambda P: (P < 1).astype(float)
assert math.isclose(float(E.integrated_gradients(grad_min, [3.0], n_steps=3000, method="midpoint")[0]), Q7B,
                    rel_tol=1e-3)
assert float(E.integrated_gradients(grad_min, [3.0], baseline=[2.0], n_steps=10)[0]) == Q7C

# ✏️ B6.4: grad_cam, with and without the ReLU
assert np.allclose(E.grad_cam(A_B64[None], G_B64[None])[0], NORM_B64)
assert math.isclose(float(E.grad_cam(A_B64[None], G_B64[None], relu=False)[0, 0, 2]), NORELU_TOPRIGHT_B64)
assert np.allclose(E.grad_cam(10 * A_B64[None], G_B64[None])[0], NORM_B64)
assert np.allclose(E.grad_cam(A_B64[None], 10 * G_B64[None])[0], NORM_B64)
assert np.allclose(E.grad_cam(A_B64[None], G_B64[None], relu=False, normalize=False)[0], RAW_B64)

# ✏️ B6.5: group_rates and the two gaps, from the labels rebuilt out of the counts
def labels_from_counts(counts):
    y_true, y_pred, groups = [], [], []
    for group, (tp, fn, fp, tn) in counts.items():
        for t, p, k in ((1, 1, tp), (1, 0, fn), (0, 1, fp), (0, 0, tn)):
            y_true += [t] * k
            y_pred += [p] * k
            groups += [group] * k
    return np.array(y_true), np.array(y_pred), np.array(groups)


y_true, y_pred, groups = labels_from_counts(COUNTS_B65)
rates = E.group_rates(y_true, y_pred, groups)
for g in ("F", "H"):
    for ours, theirs in (("base", "base_rate"), ("sel", "selection_rate"), ("tpr", "tpr"), ("fpr", "fpr"),
                         ("ppv", "precision"), ("acc", "accuracy")):
        assert math.isclose(rates[g][theirs], R_B65[g][ours])
assert math.isclose(E.demographic_parity_difference(y_pred, groups), DP_B65)
assert math.isclose(E.equalized_odds_difference(y_true, y_pred, groups), EO_B65)
from sklearn.metrics import confusion_matrix as sk_confusion_matrix

for g, (tp, fn, fp, tn) in COUNTS_B65.items():
    assert (sk_confusion_matrix(y_true[groups == g], y_pred[groups == g]) == [[tn, fp], [fn, tp]]).all()
for p_, ppv, fnr, fpr in ((R_B65[g]["base"], R_B65[g]["ppv"], 1 - R_B65[g]["tpr"], R_B65[g]["fpr"]) for g in "FH"):
    assert math.isclose(p_ / (1 - p_) * (1 - ppv) / ppv * (1 - fnr), fpr)    # ∂ B6.6, question 5

# recall R3: the confusion matrix of ch. 3 (the reference metrics.py) and scikit-learn
y3 = np.repeat([0, 0, 1, 1], [TN_R3, FP_R3, FN_R3, TP_R3])
p3 = np.repeat([0, 1, 0, 1], [TN_R3, FP_R3, FN_R3, TP_R3])
assert (np.asarray(mylearn.metrics.confusion_matrix(y3, p3)) == CM_R3).all()
assert (sk_confusion_matrix(y3, p3) == CM_R3).all()
r3 = E.group_rates(y3, p3, np.zeros(len(y3), int))[0]
assert math.isclose(r3["tpr"], R3A) and math.isclose(r3["fpr"], R3B) and math.isclose(r3["precision"], R3C)
assert math.isclose(r3["selection_rate"], R3D)

# quiz Q8: the sizes of the maps of an untrained ResNet-18 (no download)
try:
    import torch
    from torchvision.models import resnet18
except ImportError:
    print("ℹ️ torchvision is not installed: the sizes of quiz Q8 are not checked")
else:
    net, sizes = resnet18(weights=None).eval(), {}
    hooks = [getattr(net, name).register_forward_hook(lambda m, i, o, name=name: sizes.__setitem__(name, o.shape[-1]))
             for name in ("layer1", "layer4")]
    with torch.no_grad():
        net(torch.zeros(1, 3, 160, 160))
        assert sizes["layer4"] == Q8A
        net(torch.zeros(1, 3, 224, 224))
        assert sizes["layer1"] == Q8C
    for hook in hooks:
        hook.remove()
print("✅ the paper answers agree with mylearn (reference: explain, metrics), shap, scikit-learn and torchvision")

✅ the paper answers agree with mylearn (reference: explain, metrics), shap, scikit-learn and torchvision


**B6.Q1 — Interprétable par construction ou expliqué après coup ?**

In [3]:
wb.record("B6.Q1a", "B", mistakes={"relis « Interprétable par construction, ou expliqué après coup » (§206.1) : la forêt se lit-elle elle-même, ou faut-il TreeSHAP ?": "A", "une carte de saillance est-elle le modèle, ou une explication ajoutée ? Relis 206.1": "C"})
wb.record("B6.Q1b", False, mistakes={"relis la définition de l'explication post hoc (§206.1) : est-elle la règle du modèle ?": True})
wb.record("B6.Q1c", "A", mistakes={"Grad-CAM se contente-t-il des prédictions ? Relis la phrase sur les méthodes spécifiques (§206.1)": "B", "Grad-CAM change-t-il le modèle ? Relis « Interprétable par construction, ou expliqué après coup »": "C"})
wb.record("B6.Q1d", True, mistakes={"relis 206.1 : de quoi l'importance par permutation a-t-elle besoin ?": False})
wb.record("B6.Q1e", Q1E, mistakes={"c'est la profondeur : on demande le nombre de feuilles": 4,
          "c'est le nombre de nœuds internes : on demande les feuilles": 15,
          "c'est le nombre de feuilles d'un arbre de profondeur 3": 8,
          "c'est le nombre total de nœuds : on demande les feuilles": 31})

WB_ANSWER {"hash": "d307e0a097e391aa7798f9e9e626ad0f7125d4d07a0b655a4b5cfc854874ecfd", "id": "B6.Q1a", "kind": "str", "mistakes": {"aab24fcb0223aa17921f4001e524f48aaa4069357ee46ae56506ea678e5ce8ba": "une carte de saillance est-elle le modèle, ou une explication ajoutée ? Relis 206.1", "aba54b0fc3893e53492c55cdc2e8cfc819c20c47d41b369e195be2a32215377c": "relis « Interprétable par construction, ou expliqué après coup » (§206.1) : la forêt se lit-elle elle-même, ou faut-il TreeSHAP ?"}}
📝 Ex B6.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "888df880ca75b060eb3390fe7949b4d0338af128fdd5dc96142a769ce53bc394", "id": "B6.Q1b", "kind": "bool", "mistakes": {"d6ebe05f27bacb858c5a38d34d734fe71c2250e85dd36bd2b46eab873f6c80e2": "relis la définition de l'explication post hoc (§206.1) : est-elle la règle du modèle ?"}}
📝 Ex B6.Q1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "077f54c8fb60d28dfffe255fc5dbf1cc6a015d7c29fd19c83da5cf603483cc64", "id": "B6.Q1c", "kind": "str", "mistakes": {"1aa58b

**B6.Q2 — Explication globale ou locale : quel outil répond à quelle question ?**

In [4]:
wb.record("B6.Q2a", "C", mistakes={"la question porte-t-elle sur tout le modèle ou sur un dossier ? Relis « Global ou local » (§206.1)": "A", "la question compare-t-elle des groupes ? Relis le tableau de « Global ou local »": "B"})
wb.record("B6.Q2b", "A", mistakes={"un waterfall explique combien de prédictions ? Relis le tableau de « Global ou local »": "B", "une courbe ICE d'un seul client dit-elle ce que le modèle utilise en tout ? Relis 206.2": "C"})
wb.record("B6.Q2c", False, mistakes={"relis la fin de « Global ou local » (§206.1) : une importance moyenne dit-elle tout d'un cas particulier ?": True})
wb.record("B6.Q2d", "B", mistakes={"une moyenne sur 1 000 clients explique-t-elle encore un client ? Relis la puce sur le beeswarm (§206.4)": "A", "la moyenne des |φ_i| mélange-t-elle une colonne ? Relis la puce sur le beeswarm (§206.4)": "C"})
wb.record("B6.Q2e", Q2E, decimals=2, mistakes={"c'est le score après mélange : l'importance est une baisse": 0.64,
          "fais la moyenne des cinq mélanges avant de soustraire": 0.83 - 0.66,
          "vérifie le sens de la soustraction": -Q2E})

WB_ANSWER {"hash": "987051d19ffe9e58c2c1215b10e88812cc7a6d2b7d4a619100d878d416b7fd28", "id": "B6.Q2a", "kind": "str", "mistakes": {"0ddcbf07a3d8ab2298113a0fc67151a4f0631fc713ab194f0e32d37de7b84aa3": "la question porte-t-elle sur tout le modèle ou sur un dossier ? Relis « Global ou local » (§206.1)", "21dc7ef786832d7c6c15ce5ef4b51274ad7480b2645ed2f777fd65393fd2900c": "la question compare-t-elle des groupes ? Relis le tableau de « Global ou local »"}}
📝 Ex B6.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "68308a003cc4a62b849548f141375c7c808076ac0a79370f59883963ddd60647", "id": "B6.Q2b", "kind": "str", "mistakes": {"1fc295f487388870b01a78c4ad6dae9c4bab6c775c2b7fb180d3aea7db493d7e": "une courbe ICE d'un seul client dit-elle ce que le modèle utilise en tout ? Relis 206.2", "21935d1d96b1f50c2a8bd7f67bf2d5726845597eda3efe5552053de0a2680d30": "un waterfall explique combien de prédictions ? Relis le tableau de « Global ou local »"}}
📝 Ex B6.Q2b : réponse enregistrée (str).
WB_ANSWER {"has

**B6.Q3 — Dépendance partielle et courbes ICE : ce que montre la moyenne, ce qu'elle cache**

In [5]:
wb.record("B6.Q3a", "C", mistakes={"fais la moyenne de v · x₂ sur tous les exemples : que vaut la moyenne de x₂ ?": "A", "relis la formule de PD_j(v) : on moyenne v · x₂ sur tous les exemples ; que vaut la moyenne de x₂ ?": "B"})
wb.record("B6.Q3b", 2, mistakes={"chaque exemple a sa courbe ICE : quelles pentes prennent-elles selon x₂ ?": 1,
          "une courbe ICE de ce modèle est une droite : combien de valeurs de x₂ donnent-elles des pentes différentes ?": 4})
wb.record("B6.Q3c", Q3C, decimals=2, mistakes={"tu as mis la moyenne de x₂ au carré : la dépendance partielle fait la moyenne des prédictions": 6 + 1.0,
          "relis la question : on impose v = 2 à x₁ ; n'oublie pas le terme x₂²": 6.0,
          "fais la moyenne de x₂² sur les trois valeurs, pas leur somme": 6 + 5.0})
wb.record("B6.Q3d", True, mistakes={"relis le paragraphe qui suit le code de 206.2 : que fait centered=True à chaque courbe ?": False})
wb.record("B6.Q3e", "A", mistakes={"relis la formule de PD_j(v) : sur quels exemples fait-on la moyenne ?": "B", "relis le piège classique « une dépendance partielle hors des données » (§206.2)": "C"})

WB_ANSWER {"hash": "b2c19e700fda3040689b8136cd689d0743c3e2869c886dbbd3165e42b319e582", "id": "B6.Q3a", "kind": "str", "mistakes": {"20f8338c9a62ffa334f6f81c30dc472c0f3d395650470834b4e3d77e9f1abe32": "relis la formule de PD_j(v) : on moyenne v · x₂ sur tous les exemples ; que vaut la moyenne de x₂ ?", "c5b1ddded7fda984795b86ac3d86b490d1b20fe5743081f46876fde8f4d23e76": "fais la moyenne de v · x₂ sur tous les exemples : que vaut la moyenne de x₂ ?"}}
📝 Ex B6.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "db095900c78223a3f157b58f86d65fce3780263c1f24ba4032623aed1dd0efb7", "id": "B6.Q3b", "kind": "int", "magnitude": 0, "mistakes": {"056e1a805897a185a32561f16071972160a81f1e02e0c7c15fc46f9e0b8dae97": "chaque exemple a sa courbe ICE : quelles pentes prennent-elles selon x₂ ?", "7a62308115ea6765c0321391f7e0edbae01a470a757fd877e03ab5b08e4602e6": "une courbe ICE de ce modèle est une droite : combien de valeurs de x₂ donnent-elles des pentes différentes ?"}, "sign": 1}
📝 Ex B6.Q3b : réponse e

**B6.Q4 — Les quatre propriétés de Shapley : efficacité, symétrie, joueur nul, additivité**

In [6]:
wb.record("B6.Q4a", Q4A, decimals=2, mistakes={"l'efficacité répartit f(x) moins la valeur de base, pas f(x)": 0.72 - 0.15 + 0.05,
          "vérifie le signe de φ₂ : il est négatif": 0.72 - 0.40 - 0.15 - 0.05,
          "c'est l'écart à répartir entre les trois features : retire les deux valeurs connues": 0.72 - 0.40})
wb.record("B6.Q4b", "A", mistakes={"l'efficacité porte sur la somme des parts : relis les quatre propriétés (§206.3)": "B", "la symétrie compare deux joueurs : relis les quatre propriétés (§206.3)": "C"})
wb.record("B6.Q4c", Q4C, decimals=1, mistakes={"relis l'additivité : les parts s'ajoutent, elles ne se moyennent pas": 0.1,
          "vérifie le signe de la part de h": 0.4})
wb.record("B6.Q4d", False, mistakes={"la symétrie exige que les deux joueurs apportent la même chose à toute coalition : avec x₁ = 1 et x₂ = 5, est-ce le cas ? Essaie f = x₁ + x₂": True})
wb.record("B6.Q4e", Q4E, mistakes={"c'est le nombre de toutes les coalitions : retire celles qui contiennent i": 32,
          "c'est le nombre d'ordres d'arrivée des cinq features": 120,
          "relis l'encadré 🧮 de 206.3 : un ensemble de m éléments a 2^m sous-ensembles": 5})

WB_ANSWER {"decimals": 2, "hash": "76e832763baf317b93f361f0ef891887d03136c2a06f02d8e0b0703b2cd78d08", "hash_coarse": "27cf2e8bc322a323c61c19a2b8d40b6628e5952a9f758a85c07a35178914222e", "id": "B6.Q4a", "kind": "float", "magnitude": -1, "mistakes": {"1267fec5f0661ff70dec900872dbc80acc913b48c019279c30a681fab8bb8b7f": "vérifie le signe de φ₂ : il est négatif", "30a1bc89e93871d22b6e6112761ec2be8f190da4bc3481f5c2b03d63e156ea5e": "c'est l'écart à répartir entre les trois features : retire les deux valeurs connues", "4bf4aa8d295086cc127848f9791c8e275c311a4717634a8a90c3bfd91f92ffd5": "l'efficacité répartit f(x) moins la valeur de base, pas f(x)"}, "sign": 1}
📝 Ex B6.Q4a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "d48253a44bdd2894d3199fa9cbf1350b0ac70a6e67d61a980788f2b519f884bd", "id": "B6.Q4b", "kind": "str", "mistakes": {"82ede411c6b75aac65b47f4fd8ea9ac486e8338e07e8bed5e5d7c83fe935e7f5": "l'efficacité porte sur la somme des parts : relis les quatre propriétés (§206.3)", "

**B6.Q5 — Changer le fond (background) change les valeurs SHAP : vrai ou faux**

In [7]:
wb.record("B6.Q5a", False, mistakes={"relis l'efficacité avec un fond (§206.3) : de quoi dépend la somme des φ_i ?": True})
wb.record("B6.Q5b", "C", mistakes={"relis la définition de la valeur de base (§206.3) : sur quelles lignes la moyenne est-elle prise ?": "A", "relis la définition de la valeur de base (§206.3) : v(∅) est-il nul ?": "B"})
wb.record("B6.Q5c", Q5C, mistakes={"vérifie le sens (x₁ moins la moyenne du fond) et le fond utilisé : le fond A": -Q5C,
          "n'oublie pas le coefficient w₁ = 2": 1})
wb.record("B6.Q5d", Q5D, mistakes={"vérifie le sens (x₁ moins la moyenne du fond) et le fond utilisé : le fond B": -Q5D,
          "n'oublie pas le coefficient w₁ = 2": -1})
wb.record("B6.Q5e", False, mistakes={"relis les deux modes de TreeExplainer (§206.4) : répondent-ils à la même question ?": True})
wb.record("B6.Q5f", 100, mistakes={"relis le paragraphe qui suit le code de shap (§206.3) : que fait max_samples par défaut ?": 300})

WB_ANSWER {"hash": "83937823056dccc30526dc397fb2b82798ff2d711b8e47ec366852b596f0e082", "id": "B6.Q5a", "kind": "bool", "mistakes": {"b94aae08d70594877ae1908eca7bec3b2e4aa9c4c9a25ebcada3b43ba78fa98e": "relis l'efficacité avec un fond (§206.3) : de quoi dépend la somme des φ_i ?"}}
📝 Ex B6.Q5a : réponse enregistrée (bool).
WB_ANSWER {"hash": "9a625fef37a247135a768dff5c8cc47f3c965dfa6e667193c9ace9038e655940", "id": "B6.Q5b", "kind": "str", "mistakes": {"82c5d6bf137a77b1dd0fdc1b8b45048c06ae6daf2e96d6a36c8e907067159867": "relis la définition de la valeur de base (§206.3) : sur quelles lignes la moyenne est-elle prise ?", "ad4797969c8d0003d2fc6a7e234be83da75e03d5e1829b5843d22ac57303fe0a": "relis la définition de la valeur de base (§206.3) : v(∅) est-il nul ?"}}
📝 Ex B6.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "f7329948bd7ec9075d4e1b6b563ae80a348a5abbfa5ef8c21cbb384c7e412c9e", "id": "B6.Q5c", "kind": "int", "magnitude": 0, "mistakes": {"1b66b189aecb1f1df91a7531f708b846a01803a002a1f

**B6.Q6 — Features corrélées : ce que SHAP répartit et ce qu'il ne dit pas**

In [8]:
wb.record("B6.Q6a", round(Q6A), mistakes={"relis le cas linéaire : φ₁ = w₁ (x₁ − moyenne), avec w₁ = 2": 4,
          "n'oublie pas de retirer la moyenne du fond": 3})
wb.record("B6.Q6b", round(Q6B), mistakes={"le modèle B lit-il x₂ ? Relis le cas linéaire": 2,
          "le modèle B lit-il x₂ ? Relis « Features corrélées : ce que SHAP répartit »": 4})
wb.record("B6.Q6c", True, mistakes={"calcule φ₁ + φ₂ pour les deux modèles, avec la formule du cas linéaire": False})
wb.record("B6.Q6d", "C", mistakes={"relis « Features corrélées : ce que SHAP répartit » : le Shapley observationnel est-il fidèle au modèle ou aux données ?": "A", "relis « Features corrélées : ce que SHAP répartit » : pour qui le Shapley observationnel a-t-il été conçu ?": "B"})
wb.record("B6.Q6e", "A", mistakes={"la plus grande des deux dépend du modèle choisi : relis la fin de « Features corrélées »": "B", "la répartition entre deux copies est-elle stable ? Relis la fin de « Features corrélées »": "C"})

WB_ANSWER {"hash": "164b90b2fa07b6e4b937d60f5f34eaddf9bfb77ac6f3b718fa86cb0610e61505", "id": "B6.Q6a", "kind": "int", "magnitude": 0, "mistakes": {"7a471a0b00af967f000fa31a86860d4ea8b9ff5cae7432c7ebc4da0829606f8a": "relis le cas linéaire : φ₁ = w₁ (x₁ − moyenne), avec w₁ = 2", "e8073e4d86670137e5bc1172f12ef54c28bc223f9624394f2254ca8b239d5d8b": "n'oublie pas de retirer la moyenne du fond"}, "sign": 1}
📝 Ex B6.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "2ac4feb0441c2a24c31d0d79b0367ed6dc8c040cff923ac4e5ab661defaac0b9", "id": "B6.Q6b", "kind": "int", "magnitude": null, "mistakes": {"9871a8c8caf5fd7516960e2a4168d5c0825d4fc5828a3c0a3abdbe6f3925213b": "le modèle B lit-il x₂ ? Relis « Features corrélées : ce que SHAP répartit »", "ba2f62670d10596b9043c96bbc2518509f8bea799b5d69743b5848c3684350c2": "le modèle B lit-il x₂ ? Relis le cas linéaire"}, "sign": 0}
📝 Ex B6.Q6b : réponse enregistrée (int).
WB_ANSWER {"hash": "f4ef9ddceac6c8d1c54d3fc2573f765d6792b52c61a7941937087835c9e638a4", "

**B6.Q7 — Gradient, gradient × entrée, gradients intégrés : pourquoi une ligne de référence ?**

In [9]:
wb.record("B6.Q7a", Q7A, mistakes={"au-delà de 1, f vaut toujours 1 : quelle est sa pente ?": 1,
          "relis la question : on demande la dérivée, pas la valeur de f": 3})
wb.record("B6.Q7b", round(Q7B), mistakes={"le gradient est nul en x, mais pas tout le long du chemin depuis 0 : relis la formule des gradients intégrés": 0,
          "c'est x − x′ : multiplie par la moyenne de la pente le long du chemin": 3})
wb.record("B6.Q7c", Q7C, mistakes={"le chemin va de 2 à 3 : quelle est la pente de f sur ce segment ?": 1,
          "relis la question : la référence a changé": 3})
wb.record("B6.Q7d", True, mistakes={"pour f = wᵀx, le gradient est le même en tout point : calcule les deux attributions": False})
wb.record("B6.Q7e", "A", mistakes={"un pixel égal à sa référence a-t-il un x_i − x′_i non nul ? Relis la formule des gradients intégrés": "B", "relis la formule des gradients intégrés : que vaut le facteur x_i − x′_i pour un pixel du fond ?": "C"})

WB_ANSWER {"hash": "073b012e8dfbf6570edb638cea2f28b08d566e57ca374504f067016b30200298", "id": "B6.Q7a", "kind": "int", "magnitude": null, "mistakes": {"c6867804ba0ffa1c1c56ba0c3d659373749d40c436a846642aff64e0a116765d": "relis la question : on demande la dérivée, pas la valeur de f", "f1513c5af6f865527a5c0c7e3623f713b550b5c41b8f21ee7b65b01cbd8b199e": "au-delà de 1, f vaut toujours 1 : quelle est sa pente ?"}, "sign": 0}
📝 Ex B6.Q7a : réponse enregistrée (int).
WB_ANSWER {"hash": "b5e7baf49ceb0fe0d0506e299d91053d913171a46d33e985a2d4b7e234b29778", "id": "B6.Q7b", "kind": "int", "magnitude": 0, "mistakes": {"083d7c76afed677c2e6f106a8113cd9afe4b1032656769ba356f63a7e8f39865": "c'est x − x′ : multiplie par la moyenne de la pente le long du chemin", "bc3606b3a12a73cce6b9872878d2376cbb39acd4fc9bb945b41899e85e4d5fd1": "le gradient est nul en x, mais pas tout le long du chemin depuis 0 : relis la formule des gradients intégrés"}, "sign": 1}
📝 Ex B6.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash

**B6.Q8 — Grad-CAM : pourquoi la dernière couche convolutive ?**

In [10]:
wb.record("B6.Q8a", Q8A, mistakes={"c'est la taille de layer3, qui divise par 16 : relis les quatre détails de « Avec PyTorch : deux hooks »": 160 // 16,
          "c'est une division par 8 : de combien layer4 divise-t-il la taille de l'image ?": 160 // 8,
          "c'est la taille de layer1 : relis la question": 160 // 4})
wb.record("B6.Q8b", "B", mistakes={"relis « Pourquoi la dernière couche convolutive ? » (§206.6) : que vient-il après elle ?": "A", "une softmax rend un vecteur de probabilités, sans position : relis « Pourquoi la dernière couche convolutive ? »": "C"})
wb.record("B6.Q8c", Q8C, mistakes={"relis l'énoncé : layer1 divise par 4": 224 // 8,
          "c'est la taille de layer4 à 224 pixels : relis la question": 7,
          "relis l'énoncé : layer1 divise par 4, pas par 2": 112})
wb.record("B6.Q8d", True, mistakes={"un poids α_c peut-il être négatif ? Relis l'exemple à la main de 206.6": False})
wb.record("B6.Q8e", "C", mistakes={"relis la phrase qui suit la formule de Grad-CAM (§206.6) : de quel score prend-on le gradient ?": "A", "relis la phrase qui suit la formule de Grad-CAM (§206.6) : avant ou après la softmax ?": "B"})

WB_ANSWER {"hash": "ea16ef9493c4f7015cf90b8d7026b811cd61cdc929e6d1a1474cfb0bd72e49d2", "id": "B6.Q8a", "kind": "int", "magnitude": 0, "mistakes": {"92e379ffd9d8af1e3380e9467076983130439688621498d834c514cff2970dd1": "c'est la taille de layer1 : relis la question", "b98232444f3d19733d2466b5dfc7ed767ad82bec659f5473964eb731b5a8d70b": "c'est une division par 8 : de combien layer4 divise-t-il la taille de l'image ?", "e220f71964d6826dc1379414d2c63e575138676a347afcd8f0742c6196566fc7": "c'est la taille de layer3, qui divise par 16 : relis les quatre détails de « Avec PyTorch : deux hooks »"}, "sign": 1}
📝 Ex B6.Q8a : réponse enregistrée (int).
WB_ANSWER {"hash": "90ac7bc1ded65071349ef3e44a1b3fd83f0f7c76649b10222150e52f031d65a0", "id": "B6.Q8b", "kind": "str", "mistakes": {"116f2f8eb264adc0f3c97c79e22776571c79ecba21aa2f92b3ae9a8bff1c5531": "une softmax rend un vecteur de probabilités, sans position : relis « Pourquoi la dernière couche convolutive ? »", "ab91eb640cf4a7c1c11c9d27a5dbb879940bcb42

**B6.Q9 — Une carte de saillance qui ne dépend pas du modèle : que prouve le test de randomisation ?**

In [11]:
wb.record("B6.Q9a", "B", mistakes={"relis « Les tests de santé » (§206.7) : qu'exige-t-on d'une explication quand le modèle change ?": "A", "le test de randomisation mesure-t-il la robustesse aux attaques ? Relis 206.7": "C"})
wb.record("B6.Q9b", False, mistakes={"relis le paragraphe « Ce que prouve un test de santé réussi » (§206.7)": True})
wb.record("B6.Q9c", "C", mistakes={"c'est la randomisation indépendante : relis les deux variantes (§206.7)": "A", "c'est la randomisation des données : relis les deux tests (§206.7)": "B"})
wb.record("B6.Q9d", False, mistakes={"une corrélation de rangs compare l'ordre des pixels : deux cartes peuvent-elles avoir le même ordre et des valeurs différentes ?": True})
wb.record("B6.Q9e", Q9E, decimals=1, mistakes={"relis l'énoncé : CIFAR-10 a dix classes, et les labels du test ne sont pas mélangés": 0.5,
          "un réseau qui répond au hasard a-t-il une accuracy nulle ?": 0.0,
          "c'est l'accuracy d'entraînement d'un réseau qui apprend par cœur : relis la question, on demande le test": 1.0})

WB_ANSWER {"hash": "e1c36fb877bc0e08a330ab5827d8e1d84968583285016b96fa3c44333eb407e1", "id": "B6.Q9a", "kind": "str", "mistakes": {"21ea56fb3e0f78896e6529f5e263adfd55f673e6c1c36545075ba176773fe5cb": "le test de randomisation mesure-t-il la robustesse aux attaques ? Relis 206.7", "be221d27e2a843d29c32e665bf589a3e1e1375ed0c5d1ce9e2c2fbf1520dc216": "relis « Les tests de santé » (§206.7) : qu'exige-t-on d'une explication quand le modèle change ?"}}
📝 Ex B6.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "5c5a791c21648bc02a2ee4c6664f6a51442e2bfd72e7f9fbdccd7a5fa84d159c", "id": "B6.Q9b", "kind": "bool", "mistakes": {"5b58d67d76e93b32d6f101a2b4708be078806bf6447f4b910fde1404a37a6d8a": "relis le paragraphe « Ce que prouve un test de santé réussi » (§206.7)"}}
📝 Ex B6.Q9b : réponse enregistrée (bool).
WB_ANSWER {"hash": "f8fd2a56516a1b1021a213957a65eca3246c723b4e5858b2f877648e2bba12db", "id": "B6.Q9c", "kind": "str", "mistakes": {"44b864ea4d714828b68258bfedc3e7342c349372bbd37d8140dd437a09d3b

**B6.Q10 — Sources de biais : données, labels, variables de substitution, boucles de rétroaction**

In [12]:
wb.record("B6.Q10a", "A", mistakes={"les décisions du modèle fabriquent-elles ses données ? Relis les puces de « D'où vient un biais » (§206.8)": "B", "les transcriptions sont-elles fausses ? Relis les puces de « D'où vient un biais » (§206.8)": "C"})
wb.record("B6.Q10b", "B", mistakes={"la note des managers est-elle une feature ou la cible ? Relis 206.8": "A", "le modèle sert-il une autre population que celle de son évaluation ? Relis 206.8": "C"})
wb.record("B6.Q10c", "C", mistakes={"qui choisit les dossiers contrôlés ? Relis les puces de « D'où vient un biais »": "A", "une colonne porte-t-elle un attribut sensible à sa place ? Relis la question : qui choisit les contrôles ?": "B"})
wb.record("B6.Q10d", True, mistakes={"une colonne qui n'est pas le sexe peut-elle en porter une part de l'information ? Relis la puce sur les variables de substitution (§206.8)": False})
wb.record("B6.Q10e", "A", mistakes={"le modèle sert-il une autre population ? Relis les puces de 206.8 : qu'est-ce qui est mal mesuré ?": "B", "les décisions du modèle changent-elles le capteur ? Relis les puces de 206.8": "C"})

WB_ANSWER {"hash": "1c53b2cdf5b24fcd37720ce299fc427a9d5198375b0f23157e5c3c1fc5cb4d69", "id": "B6.Q10a", "kind": "str", "mistakes": {"bca4c37888c1e1cc4740410fcf08edb9501b628be68167012a2d73872abb7bf8": "les transcriptions sont-elles fausses ? Relis les puces de « D'où vient un biais » (§206.8)", "ebf36763d20b7a2295b5ef38cc74413458f6f005c0e3b03691237aed59c61886": "les décisions du modèle fabriquent-elles ses données ? Relis les puces de « D'où vient un biais » (§206.8)"}}
📝 Ex B6.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "3fb573d6edace8e9a5720ff43d249311f1dfa85171d738965224f469cfd00ca4", "id": "B6.Q10b", "kind": "str", "mistakes": {"07d439caa861990657a800f9bde6e60a73091da94540e3e6fea1dd0e5b11a761": "la note des managers est-elle une feature ou la cible ? Relis 206.8", "b381cf6bb3f0637e4c1fd43ac44a59e250e75590e2d91e9d4dd5945174f7dc9c": "le modèle sert-il une autre population que celle de son évaluation ? Relis 206.8"}}
📝 Ex B6.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash":

**B6.Q11 — Parité démographique, égalité des chances, calibration : peut-on tout avoir ?**

In [13]:
wb.record("B6.Q11a", Q11A, decimals=2, mistakes={"un classifieur parfait sélectionne-t-il tout le monde ? Son taux de sélection vaut le taux de base": 0.0,
          "relis la définition de Δ_DP : la différence des taux de sélection, pas leur rapport": 0.6 / 0.25})
wb.record("B6.Q11b", True, mistakes={"calcule le TPR et le FPR d'un classifieur parfait dans chaque groupe": False})
wb.record("B6.Q11c", True, mistakes={"calcule la precision d'un classifieur parfait dans chaque groupe": False})
wb.record("B6.Q11d", Q11D, decimals=2, mistakes={"relis la relation de Chouldechova : le dernier facteur est 1 − FNR": 0.5 / 0.5 * 0.25 * 0.2,
          "relis la relation : (1 − PPV) / PPV, pas PPV / (1 − PPV)": 0.5 / 0.5 * 4 * 0.8,
          "relis la relation : le premier facteur est p / (1 − p)": 0.5 * 0.25 * 0.8})
wb.record("B6.Q11e", "A", mistakes={"la parité démographique tient-elle compte de qui rembourserait ? Relis « Les critères » (§206.9)": "B", "un faux positif est ici un prêt accordé à qui ne rembourserait pas : est-ce l'erreur la plus coûteuse pour les demandeurs ?": "C"})

WB_ANSWER {"alt_coarse": ["4bfdd93a5db25683a6c6ef76d80c87a8e916819b2ece1a59cb217df8fa0e16a6"], "decimals": 2, "hash": "15e622ba2989402651cb3d364c6c13f65e02b673dce4a27ccb2a8ed2b1ee8f7d", "hash_coarse": "ffb415078ce75fe1764041e52a2a0819ce148291452f2e31467c65ecfadb6e58", "id": "B6.Q11a", "kind": "float", "magnitude": -1, "mistakes": {"46cf0e5cfbec76bb67259e14c9f162de7e425d2a3f437fff26624da7ca1da7f9": "un classifieur parfait sélectionne-t-il tout le monde ? Son taux de sélection vaut le taux de base", "5c631a6f1d297641a32ed1f2c8bdcf5f84cacd68a22c4382ff887cecb92f2b8b": "relis la définition de Δ_DP : la différence des taux de sélection, pas leur rapport"}, "sign": 1}
📝 Ex B6.Q11a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "a206040ccc70f897a12598003f328fe83247055de73fb05b44acecc2010a9f8b", "id": "B6.Q11b", "kind": "bool", "mistakes": {"27b3bff27f8ff9321bbc7de656b44719a251c93ebc61b3256395611e07f0f4b5": "calcule le TPR et le FPR d'un classifieur parfait dans chaque groupe"

**B6.Q12 — AI Act : dans quelle catégorie de risque tombe un scoring de crédit ?**

In [14]:
wb.record("B6.Q12a", "B", mistakes={"relis la puce « haut risque » de « L'AI Act » (§206.11) : que vise le point 5 b) ?": "A", "le scoring de crédit figure-t-il parmi les pratiques interdites ? Relis « L'AI Act » (§206.11)": "C"})
wb.record("B6.Q12b", "A", mistakes={"relis le point 5 b) cité en 206.11 : quelle exception contient-il ?": "B", "la détection de fraude figure-t-elle à l'article 5 ? Relis « L'AI Act »": "C"})
wb.record("B6.Q12c", "C", mistakes={"relis la puce « risque inacceptable » de « L'AI Act » (§206.11)": "A", "l'article 50 parle de transparence envers l'utilisateur : relis la puce « risque inacceptable »": "B"})
wb.record("B6.Q12d", False, mistakes={"relis l'encadré 🕰️ sur la régulation (§206.11) : quelles dates le règlement 2026/1744 a-t-il fixées ?": True})
wb.record("B6.Q12e", "B", mistakes={"relis le dernier paragraphe de « L'AI Act » : qui, du fournisseur ou du déployeur, réalise l'analyse d'impact ?": "A", "relis le dernier paragraphe de « L'AI Act » : l'article 27 s'adresse à qui ?": "C"})
wb.record("B6.Q12f", False, mistakes={"relis le texte du point 5 b) cité en 206.11 : de quelles personnes évalue-t-il la solvabilité ?": True})

WB_ANSWER {"hash": "e97b77884b2a335e11a55d033c9d53f2805eb3c6bcd4a71d5eb1494fe8aed0c8", "id": "B6.Q12a", "kind": "str", "mistakes": {"172a4c1802d2b21a8ea4aa71b3a943814141270caea477dd5b8942b8b66cc7b3": "le scoring de crédit figure-t-il parmi les pratiques interdites ? Relis « L'AI Act » (§206.11)", "23b5bb6f1541a005393a7f8f744b540358e757fb3943fa770ec3e44602728e7b": "relis la puce « haut risque » de « L'AI Act » (§206.11) : que vise le point 5 b) ?"}}
📝 Ex B6.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "c413ec9c2b1138c725b5db22eee74918c5c95d5baf16f427c715f6efa2537d88", "id": "B6.Q12b", "kind": "str", "mistakes": {"2728cb6cfe0649c54ec5cf5d66ffff069c3bf17f0af1527e4470dda14bc576ac": "relis le point 5 b) cité en 206.11 : quelle exception contient-il ?", "db24199bd13f6d873494dbcd676d55187709eb54630b3dd92243df812fc282ff": "la détection de fraude figure-t-elle à l'article 5 ? Relis « L'AI Act »"}}
📝 Ex B6.Q12b : réponse enregistrée (str).
WB_ANSWER {"hash": "dda00d3c2dfbb967f50f9057d67d

**B6.R1 — Visualiser ce qu'a appris un CNN, et l'attaque FGSM**

In [15]:
wb.record("B6.R1a", R1A, decimals=2, mistakes={"ramène chaque pixel dans [0, 1] après le pas": [0.37, 1.01, 0.1],
          "vérifie le signe : FGSM monte la loss, x + ε sign(∇L)": [0.43, 0.95, 0.1],
          "FGSM ajoute ε fois le signe du gradient, pas ε fois le gradient": [0.38, 0.99, 0.1],
          "le signe de 0 vaut 0 : le troisième pixel ne bouge pas": [0.37, 1.0, 0.13]})
wb.record("B6.R1b", True, mistakes={"relis le début de 206.5 et la section FGSM du ch. 21 : les deux calculs dérivent-ils par rapport à la même chose, avec la même passe arrière ?": False})
wb.record("B6.R1c", "B", mistakes={"relis le rappel maths « le gradient par rapport à l'entrée » du ch. 21 : les poids bougent-ils ?": "A", "la maximisation d'activation a-t-elle besoin de labels ? Relis le ch. 21": "C"})
wb.record("B6.R1d", R1D, decimals=2, mistakes={"relis la formule ε Σ|w_j| du ch. 21 : il manque un facteur": 0.01 * 0.01,
          "c'est ε fois le nombre de pixels : n'oublie pas |w_j|": 0.01 * 3072,
          "relis l'énoncé : ε = 0,01 et |w_j| = 0,01": 3.072})
wb.record("B6.R1e", False, mistakes={"relis le rappel maths du ch. 21 : la montée de gradient cherche-t-elle un minimum ?": True})

WB_ANSWER {"decimals": 2, "element_coarse": [["c0b85774ed5c4e847a92f87116e1f9c0b1baa654c62f3817119c440643c34d48"], ["63ed4448f93bb8b386f1686976b4b29e54e69e250da18279d41c84a1b2976c52"], ["ac1dac572f321071b666f23f720b75f7f214f62152cb05c006d19b77b70c5a1a"]], "element_hashes": ["bd3a6d3f8c5d2376459be79908fb3e07dcdf5ac2dd481ea6b8a3c5dda0e64487", "23df0cfdfb518ab4ce4ce752a5a4875480eb16b40aac95281c0bae92c675dba3", "bd81390edde7c131ffd78360c74dd015825da1b612e992c12f1818a8e30efc24"], "hash": "9863be79db9ae7084a2d9ab6679411a63be1e60ab7d5519359168408f02b0df3", "id": "B6.R1a", "kind": "array", "mistakes": {"25782b259235d052ea075f23ceaa0e8fbcded345133780117fa4c69be37cb564": "vérifie le signe : FGSM monte la loss, x + ε sign(∇L)", "45c2ef4b5494939e99d166c813bc5d6f1b0ce4e295788dda089767887dea1a3f": "ramène chaque pixel dans [0, 1] après le pas", "8eb6ffb01fa01460736963791fb0987ad23251303128ade45210909661b1e6b4": "le signe de 0 vaut 0 : le troisième pixel ne bouge pas", "d3137444c7ef50a6b3e75d7d2e89b1

**B6.R2 — feature_importances_ contre importance par permutation**

In [16]:
wb.record("B6.R2a", "A", mistakes={"relis l'importance des features au ch. 14 : qu'arrive-t-il aux features à forte cardinalité ?": "B", "une baisse d'impureté peut-elle être négative ? Relis le ch. 14": "C"})
wb.record("B6.R2b", True, mistakes={"relis le ch. 14 : sur quelles données l'importance MDI est-elle calculée ?": False})
wb.record("B6.R2c", 1, mistakes={"relis le ch. 14 : les importances MDI sont normalisées": 100})
wb.record("B6.R2d", "B", mistakes={"mélanger un identifiant fait-il perdre quelque chose sur des exemples jamais vus ? Relis le ch. 14": "A", "une importance par permutation fortement négative voudrait dire que le mélange aide beaucoup : est-ce plausible ?": "C"})
wb.record("B6.R2e", R2E, mistakes={"c'est la baisse d'accuracy : multiplie-la par le nombre d'exemples": 0,
          "vérifie l'ordre de grandeur : 2 points de 400 exemples": 80})

WB_ANSWER {"hash": "1e04cefd87cdae3468b2fd5dd0e7900b604686406f294808b7d0daab4a5a278b", "id": "B6.R2a", "kind": "str", "mistakes": {"d497dbfc691b90c2017409a972ea6c59bb885bc4f5db6a30d77c4ebb8c27bb15": "une baisse d'impureté peut-elle être négative ? Relis le ch. 14", "eeb8b46b7a2755cf169dafca8c7fa209c03ea1199fad5e24b7768f92dface7ec": "relis l'importance des features au ch. 14 : qu'arrive-t-il aux features à forte cardinalité ?"}}
📝 Ex B6.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "2a10c396a6b0cebe63a83863cb311850a37b1847f8703facdaf2d176d3b1cf31", "id": "B6.R2b", "kind": "bool", "mistakes": {"0c536f712973823907d97a1f7b2030cc69017b81d5a90282dc1a1b613acd8199": "relis le ch. 14 : sur quelles données l'importance MDI est-elle calculée ?"}}
📝 Ex B6.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "2e06fe06d2688aae53efcef0bbe89b9680ef720e445705f3f2a78e66a4ee0a89", "id": "B6.R2c", "kind": "int", "magnitude": 0, "mistakes": {"cda0c3fa689342cb1e6d1c30df833632b9c737328b8824291a629c41e4

**B6.R3 — TPR, FPR et precision à partir d'une matrice de confusion**

In [17]:
wb.record("B6.R3a", R3A, decimals=2, mistakes={"le TPR divise TP par les positifs réels (la ligne 1), pas par les sélectionnés": R3C,
          "le TPR divise par les positifs réels, pas par tous les exemples": 40 / 150})
wb.record("B6.R3b", R3B, decimals=2, mistakes={"c'est la part de faux positifs parmi les sélectionnés : le FPR divise par les négatifs réels": 12 / 52,
          "relis la disposition : la case [0, 1] contient quoi ?": 10 / 50,
          "le FPR divise par les négatifs réels, pas par tous les exemples": 12 / 150})
wb.record("B6.R3c", R3C, decimals=3, mistakes={"la precision divise TP par les sélectionnés (la colonne 1), pas par les positifs réels": R3A,
          "c'est la part de faux positifs parmi les sélectionnés : on demande la part de vrais positifs": 12 / 52})
wb.record("B6.R3d", R3D, decimals=3, mistakes={"c'est la part de positifs réels (la ligne 1) : on demande la part de prédictions positives": 50 / 150,
          "le taux de sélection divise par tous les exemples, pas par les négatifs réels": 52 / 100})
wb.record("B6.R3e", "A", mistakes={"relis la disposition : la ligne 0 est la vérité 0, la colonne 1 la prédiction 1": "B", "relis la disposition : la ligne 0 est la vérité 0": "C"})

WB_ANSWER {"decimals": 2, "hash": "cd85fa2c3af82613f6d2ce7f2f5635f8f05670c646dfa5415536020ae41d9468", "hash_coarse": "e82a7e7a8e620c5e1eb09052937601f2399c5ceb221a2e97b91b050a2309da98", "id": "B6.R3a", "kind": "float", "magnitude": -1, "mistakes": {"857292a4a0e75bade5e8143c630a47a0a70d5500cf192a11c62aaeb2a6ddb2df": "le TPR divise TP par les positifs réels (la ligne 1), pas par les sélectionnés", "b369d8cddb255e82855649cd540058f3f5b2211a31b875cbc330cf61a8440ace": "le TPR divise par les positifs réels, pas par tous les exemples"}, "sign": 1}
📝 Ex B6.R3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "b791dd0c4df9b6a1288c50a6c5ba8168b9789dc34fefb46b0beea231cc5c9800", "hash_coarse": "8bd62e18b8f3459e2ad6bd3d5d2740cffc66cbc99f0ff28e742eb65da385043d", "id": "B6.R3b", "kind": "float", "magnitude": -1, "mistakes": {"5b321aad69e7c89faaa16efb93b32ed7b5a16f2e33559aa6fd08ec5878318ecc": "c'est la part de faux positifs parmi les sélectionnés : le FPR divise par les né

**Ex B6.1 — Valeurs de Shapley exactes à la main : trois features, huit coalitions**

In [18]:
wb.record("B6.1a", round(V0_B61), mistakes={"tu as calculé f sur la ligne moyenne du fond : v(∅) est la moyenne de f sur les deux lignes": 5,
          "relis la question : v(∅) n'a aucune feature du client": 11,
          "tu as additionné les deux lignes : v(S) est une moyenne": 12})
wb.record("B6.1b", round(V1_B61), mistakes={"dans v({1}), seule la feature 1 vient du client : as-tu inversé le masque ?": round(V_B61[(0, 1, 1)]),
          "tu as additionné les deux lignes : v(S) est une moyenne": 16,
          "relis la question : v({1}) garde x₂ et x₃ du fond": 11})
wb.record("B6.1c", round(V13_B61), mistakes={"dans v({1, 3}), les features 1 et 3 viennent du client : as-tu inversé le masque ?": round(V_B61[(0, 1, 0)]),
          "tu as additionné les deux lignes : v(S) est une moyenne": 24})
wb.record("B6.1d", W3_B61, decimals=2, mistakes={"c'est le poids des coalitions de taille 0 ou 2 : {3} a un membre": 1 / 3,
          "les poids ne sont pas uniformes sur les quatre coalitions sans la feature 1 : relis la formule": 1 / 4,
          "les poids ne sont pas uniformes sur les huit coalitions : relis la formule": 1 / 8})
wb.record("B6.1e", PHI_B61[0], decimals=1, mistakes={"tu n'as gardé que l'apport de la feature 1 à la coalition vide (ou le seul terme 2x₁) : il y a quatre coalitions sans elle": FIRST_B61[0],
          "tu n'as gardé que son apport quand elle arrive la dernière : il y a quatre coalitions sans elle": LAST_B61[0]})
wb.record("B6.1f", round(PHI_B61[1]), mistakes={"vérifie le signe : x₂ est plus petit que ses valeurs du fond": 1,
          "x₂ entre dans f : un joueur nul n'apporte jamais rien, est-ce le cas ?": 0})
wb.record("B6.1g", PHI_B61[2], decimals=1, mistakes={"tu n'as gardé que l'apport de la feature 3 à la coalition vide": FIRST_B61[2],
          "tu n'as gardé que son apport quand elle arrive la dernière": LAST_B61[2],
          "as-tu pris pour valeur de base f sur la ligne moyenne du fond ? Relis a)": PHI_B61[2] + 1})
wb.record("B6.1h", False, mistakes={"calcule ce que l'interaction apporte à φ₁ (retire le terme 2(x₁ − moyenne)) et compare à φ₃": True})
wb.record("B6.1i", EVALS_B61, mistakes={"chaque v(S) demande une évaluation par ligne du fond : il y en a deux": 8,
          "chaque v(S) se calcule une fois et sert aux trois features": 3 * EVALS_B61,
          "compte toutes les coalitions, la vide et la pleine comprises": 6 * 2,
          "la coalition pleine se calcule aussi sur les deux lignes du fond, comme les autres": EVALS_B61 - 1})
wb.record("B6.1j", PHI_MEAN_B61[0], decimals=1, mistakes={"relis la question : le fond n'a plus qu'une ligne, recalcule les v(S)": 3.5,
          "tu n'as gardé que le terme 2x₁ : la feature 1 entre aussi dans l'interaction": 2.0})
wb.record("B6.1k", "B", mistakes={"relis l'encadré 📐 de la fiche : la feature 1 est la colonne 0": "A", "relis la signature de shapley_values : que rend-elle à part phi ?": "C"})

WB_ANSWER {"hash": "f58d4b2bc337b00439d9b22391ae3b71a395b9f5006dbd1676c27608765eeb6f", "id": "B6.1a", "kind": "int", "magnitude": 0, "mistakes": {"5e1a405ae8f3d5bc5642289c032237d994a27332480bb45a6203ba4565e3e446": "tu as calculé f sur la ligne moyenne du fond : v(∅) est la moyenne de f sur les deux lignes", "6af629807b890022e94ab62b31582de3b3f748726aa692f61487f1ed08647f6e": "relis la question : v(∅) n'a aucune feature du client", "f11a358b26539e304a5fd18e14f02bb35fce96f9c0fe30d93ba952cf6d01de2b": "tu as additionné les deux lignes : v(S) est une moyenne"}, "sign": 1}
📝 Ex B6.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "ce32d6a7efeee5480b7e73730b89d8f4cba2f9aa82955a4beecf43058631b58a", "id": "B6.1b", "kind": "int", "magnitude": 0, "mistakes": {"484ec8905ec483b6d9d60fdb296e044512ac04682c546b479a4bedce7eee0d35": "relis la question : v({1}) garde x₂ et x₃ du fond", "7988c915d55a3af63871c4cf097318c3732cdf84afde8cde0214f1ce9c2d7809": "dans v({1}), seule la feature 1 vient du client : a

**Ex B6.3 — Gradients intégrés à la main sur f(x) = x₁² + 3x₂ et vérification de la complétude**

In [19]:
wb.record("B6.3a", GRAD_AT_X_B63, decimals=1, mistakes={"la dérivée de x₁² est 2x₁": [2.0, 3.0],
          "la dérivée de 3x₂ par rapport à x₂ ne dépend pas de x₂": [4.0, -3.0]})
wb.record("B6.3b", GXI_B63, decimals=1, mistakes={"multiplie chaque dérivée par la valeur de sa feature": GRAD_AT_X_B63,
          "vérifie le signe : x₂ = −1": [8.0, 3.0]})
wb.record("B6.3c", round(GAP_B63), mistakes={"c'est la somme de gradient × entrée : on demande l'écart de prédiction f(x) − f(x′)": 5,
          "vérifie le signe de 3x₂ : x₂ = −1": 7,
          "vérifie le sens de la soustraction : f(x) − f(x′)": -1})
wb.record("B6.3d", IG_EXACT_B63, decimals=1, mistakes={"c'est gradient × entrée : intègre la dérivée le long du chemin": GXI_B63,
          "multiplie l'intégrale par x_i − x′_i": [2.0, 3.0],
          "vérifie le signe de la seconde attribution : x₂ = −1": [4.0, 3.0]})
wb.record("B6.3e", IG1_LEFT_B63, decimals=1, mistakes={"c'est la règle 'right' : 'left' prend les points k/n pour k = 0 à n − 1": IG1_RIGHT_B63,
          "multiplie la moyenne des gradients par x₁ − x′₁": IG1_LEFT_B63 / 2,
          "n = 5 intervalles : les points 0 ; 0,2 ; 0,4 ; 0,6 ; 0,8": float(ig_rule_b63(4, "left")[0]),
          "relis la question : la règle de gauche, avec n = 5": 4.0})
wb.record("B6.3f", IG1_RIGHT_B63, decimals=1, mistakes={"c'est la règle 'left' : 'right' prend les points k/n pour k = 1 à n": IG1_LEFT_B63,
          "multiplie la moyenne des gradients par x₁ − x′₁": IG1_RIGHT_B63 / 2,
          "relis la question : la règle de droite, avec n = 5": 4.0})
wb.record("B6.3g", IG1_TRAP_B63, decimals=1, mistakes={"relis la question : la règle des trapèzes, dont les deux extrémités comptent pour moitié": 4.8,
          "multiplie la somme pondérée par x₁ − x′₁": 2.0,
          "les poids valent 1/n, divisés par deux aux extrémités : relis l'énoncé": 2 * (0 + 0.8 + 1.6 + 2.4 + 3.2 + 2) / 6})
wb.record("B6.3h", N_LEFT_B63, mistakes={"avec ce n, l'erreur dépasse encore un peu 0,03 : prends le plus petit entier qui convient": N_LEFT_B63 - 1,
          "l'erreur de la règle de gauche sur IG₁ dépend de x₁² : recalcule-la": 34,
          "recalcule l'erreur de la règle de gauche en fonction de n : où intervient x₁ ?": 67},
          fractional="le nombre d'intervalles est entier : prends le plus petit n qui convient")
wb.record("B6.3i", round(IG1_REF2_B63), mistakes={"relis la question : la référence a changé ; un gradient pris en x seul ne suffit pas non plus": 4,
          "le chemin part de x′ : son point est x′ + α(x − x′)": 2})
wb.record("B6.3j", round(SUM_REF2_B63), mistakes={"relis la question : avec cette référence, la somme vaut f(x) − f(x′)": 1,
          "relis la question : on demande la somme des deux attributions": 3,
          "c'est la seconde attribution seule : ajoute IG₁": -3})
wb.record("B6.3k", CAPTUM_B63, decimals=2, mistakes={"c'est le trapèze de ta docstring : relis l'encadré 🕰️ de 206.6 sur captum": 4.0,
          "le facteur est n/(n + 1), pas (n + 1)/n : relis l'encadré 🕰️ de 206.6": 4.8,
          "6 points font 5 intervalles, pas 6 : relis l'énoncé": 4 * 6 / 7})
wb.record("B6.3l", "C", mistakes={"n_steps compte les intervalles dans ta librairie, pas les points": "A", "relis la question : e) utilise la règle 'left'": "B"})

WB_ANSWER {"decimals": 1, "element_hashes": ["7542e16b37ec52198a1423281a3dd748dfa524dd1afe31e397eae15313414a59", "54573c89c85d45da89c20e57e05f95a0f7facca3c13c3a97ac6cd5493050b8b1"], "hash": "013c2532476dde14890d3533cfa53c5b745457c954a0df8bf988d4ebaa249ba4", "id": "B6.3a", "kind": "array", "mistakes": {"7e410381a4b1dd7502dff2894e08a1e3e63b75f3d08205928f85500fabd044cb": "la dérivée de 3x₂ par rapport à x₂ ne dépend pas de x₂", "bc5b5c34c77e781f43caeeffed1e557a25997dbd395bde4887f1fd60c7f972c9": "la dérivée de x₁² est 2x₁"}, "shape": [2]}
📝 Ex B6.3a : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "element_hashes": ["f8a102ac9cf6dddde00f12656f771eb73e0fa5e958bbb960dbc46453722508f6", "a3d7b2e456e0696602b406d7ba635bd5ab92c6aa5d08fa867bfcd5a48d501c01"], "hash": "676f37d19116d6a7aebb0235a37d2c3f801a96c7281582be4689a505ed0a4879", "id": "B6.3b", "kind": "array", "mistakes": {"605a4f94dd9a8193f06fa46ecc5ab515eb270d041ad670e144d92b974662943c": "vérifie le signe : x₂ = −1", "

**Ex B6.4 — Grad-CAM à la main : deux cartes 3×3 et leurs gradients**

In [20]:
wb.record("B6.4a", float(ALPHA_B64[0]), decimals=1, mistakes={"α_c est la moyenne des gradients, pas leur somme": float(G_B64[0].sum()),
          "α_c est la moyenne des gradients seuls, sans les multiplier par les activations": float((G_B64[0] * A_B64[0]).mean())})
wb.record("B6.4b", float(ALPHA_B64[1]), decimals=1, mistakes={"α_c est la moyenne des gradients, pas leur somme": float(G_B64[1].sum()),
          "vérifie le signe : les gradients du canal 2 sont négatifs ou nuls": -float(ALPHA_B64[1]),
          "α_c est la moyenne des gradients seuls, sans les multiplier par les activations": float((G_B64[1] * A_B64[1]).mean())})
wb.record("B6.4c", TOPRIGHT_B64, decimals=1, mistakes={"relis la question : avant le ReLU": 0.0,
          "vérifie le signe de α₂": -TOPRIGHT_B64,
          "relis la case : en haut à droite, A¹ vaut 0 et A² vaut 2": -0.5})
wb.record("B6.4d", ZEROS_B64, mistakes={"compte aussi les cases dont la somme pondérée est négative mais proche de 0": 2,
          "une case positive n'est pas mise à 0 : compte les cases négatives de la somme pondérée": 4})
wb.record("B6.4e", CENTER_B64, decimals=2, mistakes={"n'oublie pas de diviser par la plus grande valeur de la carte": 1.0,
          "on divise par la plus grande valeur absolue, pas par la somme de la carte": float(CAM_B64[1, 1] / CAM_B64.sum())})
wb.record("B6.4f", "B", mistakes={"cherche la plus grande valeur de α₁A¹ + α₂A² après le ReLU": "A", "en haut à droite, quel est le signe de la somme pondérée ?": "C"})
wb.record("B6.4g", NORELU_TOPRIGHT_B64, decimals=2, mistakes={"sans le ReLU, la case garde son signe": -NORELU_TOPRIGHT_B64,
          "n'oublie pas de diviser par la plus grande valeur absolue": TOPRIGHT_B64,
          "relis la question : sans le ReLU, la case n'est pas mise à 0": 0.0})
wb.record("B6.4h", "C", mistakes={"calcule −0,5 A¹ + A² case par case : que vaut-elle en haut à gauche ?": "A", "calcule −0,5 A¹ + A² case par case : que vaut-elle en bas à gauche ?": "B"})
wb.record("B6.4i", False, mistakes={"multiplier les activations par 10 multiplie toute la somme pondérée : que fait la division par le maximum ?": True})
wb.record("B6.4j", round(SHIFT_B64), mistakes={"c'est la taille d'un pixel de la couche : on demande un demi-pixel": 24,
          "relis l'énoncé : la carte fait 3 × 3 et l'image 72 × 72": 36})

WB_ANSWER {"decimals": 1, "hash": "d5d3424fe18930e6f715254358fdd8f8bb5408345f451e0b333a105545c87b99", "hash_coarse": "92d3c0901681b8b530c7163fa4fc78b5bae40122489b167bc401c0e8001a722b", "id": "B6.4a", "kind": "float", "magnitude": 0, "mistakes": {"0d1ca3a40e34fed6f4e45d20f6dbd680079a1a35b16e8dd115cda68042f59116": "α_c est la moyenne des gradients, pas leur somme", "cf972a8a744fac3f3466f2645ad1b9ea130ce4f7263050e37b761a9d10328e81": "α_c est la moyenne des gradients seuls, sans les multiplier par les activations"}, "sign": 1}
📝 Ex B6.4a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"alt_coarse": ["61e491c54dcc27b3e31caf91bb1d059c1d4289f516abbb3e1d5437ca6c811e77"], "decimals": 1, "hash": "8111e9bf2824e5e9cda30d9fdafb8c14e1ecff18b66e7ee9b1de2e524b25a081", "hash_coarse": "c461c85c8eeb9d1aca6ac8f17659504c664fafde08bd9f5ae0e6cd9d8163dcf0", "id": "B6.4b", "kind": "float", "magnitude": -1, "mistakes": {"1b2e4e8ab1569bffd336678f4e2b26367dbd5e8c8e6c8b050e6d2fd7e622af23": "α_c est la moy

**Ex B6.5 — Audit à la main : deux groupes, deux matrices de confusion, cinq indicateurs**

In [21]:
wb.record("B6.5a", R_B65["F"]["base"], decimals=2, mistakes={"relis la question : on demande la part de positifs réels (TP + FN)": R_B65["F"]["sel"],
          "le taux de base divise par tout le groupe, pas par les négatifs": 120 / 380})
wb.record("B6.5b", R_B65["F"]["sel"], decimals=3, mistakes={"le taux de sélection compte tous les sélectionnés, TP et FP": 72 / 500,
          "relis la question : on demande la part sélectionnée (TP + FP)": R_B65["F"]["base"]})
wb.record("B6.5c", R_B65["F"]["tpr"], decimals=2, mistakes={"c'est le FNR : le TPR divise TP par les positifs réels": 0.4,
          "c'est la precision : le TPR divise par les positifs réels, pas par les sélectionnés": R_B65["F"]["ppv"],
          "le TPR divise par les positifs réels, pas par tout le groupe": 72 / 500})
wb.record("B6.5d", R_B65["H"]["fpr"], decimals=3, mistakes={"le FPR divise par les négatifs réels, pas par tout le groupe": 40 / 640,
          "c'est la part de vrais négatifs : le FPR compte les faux positifs": 280 / 320,
          "le FPR divise par les négatifs réels (FP + TN), pas par les sélectionnés": 40 / 296})
wb.record("B6.5e", R_B65["F"]["ppv"], decimals=3, mistakes={"la precision divise TP par les sélectionnés (TP + FP)": R_B65["F"]["tpr"],
          "c'est la part de faux positifs parmi les sélectionnés : on demande la precision": 19 / 91})
wb.record("B6.5f", DP_B65, decimals=2, mistakes={"c'est l'écart des taux de base : la parité démographique compare les taux de sélection": 0.5 - 0.24,
          "relis la question : la parité démographique compare les taux de sélection": TPR_GAP_B65,
          "les taux de sélection divisent TP + FP par l'effectif du groupe": 296 / 640 - 72 / 500})
wb.record("B6.5g", EO_B65, decimals=2, mistakes={"l'equalized odds prend le plus grand des deux écarts, TPR et FPR": FPR_GAP_B65,
          "l'equalized odds prend le plus grand des deux écarts, pas leur somme": TPR_GAP_B65 + FPR_GAP_B65,
          "l'equalized odds prend le plus grand des deux écarts, pas leur moyenne": (TPR_GAP_B65 + FPR_GAP_B65) / 2})
wb.record("B6.5h", RATIO_B65, decimals=2, mistakes={"relis la question : le taux des femmes divisé par celui des hommes": 1 / RATIO_B65,
          "relis la question : on demande le rapport, pas son complément": 1 - RATIO_B65})
wb.record("B6.5i", False, mistakes={"compare ton rapport h) à 4/5 = 0,8": True})
wb.record("B6.5j", True, mistakes={"calcule l'accuracy (TP + TN) / n de chaque groupe": False})
wb.record("B6.5k", EXTRA_B65, mistakes={"c'est le nombre total de qualifiées sélectionnées : on demande combien de plus": 96,
          "le TPR des hommes s'applique aux 120 candidates qualifiées, pas à toutes": round(0.8 * 500) - 72,
          "relis l'énoncé : le TPR à atteindre est celui des hommes": 120 - 72})

WB_ANSWER {"decimals": 2, "hash": "257ab8e6ecf3999b7af489e6c7b9a8c0587de64d467ef12bd2f247b1a3dca3c8", "hash_coarse": "98ad7fc0d5097ff1486284db0abc688b681eb3fb041ee747aa3cae1097b23b76", "id": "B6.5a", "kind": "float", "magnitude": -1, "mistakes": {"4cd31025b5f4d7abe3dae51bce5a526f74c144d0b0674a8349308110bf0592a6": "relis la question : on demande la part de positifs réels (TP + FN)", "b14f68378f3e67239b18e4c0638776b7b7c6ab80f316d576f9ce126ac5d93bec": "le taux de base divise par tout le groupe, pas par les négatifs"}, "sign": 1}
📝 Ex B6.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "a5f6e396d9cc4bd1c6bdef77291c3320a31883b46d0964f074c3489637eeaae8", "hash_coarse": "de90351fa35b36f6f37c52dfee95bd63bdbdbf5c368bfe7be7d47972bbf4f6da", "id": "B6.5b", "kind": "float", "magnitude": -1, "mistakes": {"1c0f7ddc8f212a04f1b6f15d8ba3cfe293856c32e744fc7f31918547c175aab9": "relis la question : on demande la part sélectionnée (TP + FP)", "55da6d627fab51450c5c5235f97b06

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu calculer à la main les valeurs de Shapley d'un petit jeu, avec un fond, et dire ce que le fond change ?
2. Sais-tu calculer des gradients intégrés et une carte Grad-CAM à la main, et vérifier la complétude ?
3. Sais-tu auditer un classifieur par groupe (taux de sélection, TPR, FPR, precision, écarts) et expliquer pourquoi tous les critères ne peuvent pas tenir ensemble ?
4. Sais-tu dire ce qu'un scoring de crédit implique au regard du RGPD et de l'AI Act ?

**Pour aller plus loin** : l'article de Lundberg et Lee et le livre en ligne de C. Molnar, cités dans la fiche. La suite : le bonus B7, le MLOps.

*Les exercices de code B6.12 à B6.29 (modèles tabulaires, réseaux, équité) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B6` pour obtenir le notebook complet.*